[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session3a_student.ipynb)

# Session 3a — Classification

In regression we predicted a number (a redshift). In classification we predict a category: star or galaxy, AGN or not, real transient or artifact, spiral or elliptical.

Cells marked ✏️ Exercise are for you to complete — each should take only a few minutes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score,
                             precision_recall_curve, classification_report)

CLASS_COLORS = ["C0", "C1", "C2"]

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110,
    "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False,
})

def plot_data(X, y, ax, labels=None, s=18, alpha=0.8):
    # scatter plot of 2-D data, one colour and marker per class
    markers = ["o", "s", "^"]
    for k in np.unique(y):
        name = labels[k] if labels is not None else f"class {k}"
        ax.scatter(X[y == k, 0], X[y == k, 1], s=s, alpha=alpha, color=CLASS_COLORS[k],
                   marker=markers[k], edgecolor="white", linewidth=0.4, label=name)
    ax.set(xlabel="feature 1", ylabel="feature 2")

def plot_decision_boundary(model, X, y, ax, labels=None, title="", n=300):
    # shade the regions the model assigns to each class, and draw the boundary
    pad = 0.5
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))
    grid = np.c_[gx.ravel(), gy.ravel()]
    n_classes = len(model.classes_)
    if n_classes == 2:
        p = model.predict_proba(grid)[:, 1].reshape(gx.shape)
        ax.contourf(gx, gy, p, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.4)
        ax.contour(gx, gy, p, levels=[0.5], colors="black", linewidths=2)
    else:
        pred = model.predict(grid).reshape(gx.shape)
        tints = ListedColormap(plt.cm.Pastel1.colors[:n_classes])
        ax.contourf(gx, gy, pred, levels=np.arange(n_classes + 1) - 0.5, cmap=tints)
        ax.contour(gx, gy, pred, levels=np.arange(n_classes) + 0.5, colors="black", linewidths=1.5)
    plot_data(X, y, ax, labels)
    ax.set(xlim=(x0, x1), ylim=(y0, y1), title=title)
    ax.grid(False)

## 1. From regression to classification

Our first data set has two features and two classes — think of *feature 1* and *feature 2* as, say, a colour and a size, and the classes as two kinds of object.

In [ ]:
X, y = make_blobs(n_samples=400, centers=[(-1, -1), (1.2, 1.0)], cluster_std=1.1, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

fig, ax = plt.subplots()
plot_data(X_train, y_train, ax)
ax.set_title("Two classes, two features (training set)")
ax.legend();

Could we just use linear regression, predicting $y = 0$ or $1$ and thresholding at 0.5? It sort of works, but the output is not a probability — it happily goes below 0 and above 1 — and outliers far from the boundary drag the fit around.

Logistic regression fixes this by passing the linear combination through the sigmoid function:

$$P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{w}\cdot\mathbf{x} + b), \qquad \sigma(t) = \frac{1}{1 + e^{-t}} .$$

* The output is always between 0 and 1, and we interpret it as a probability.
* Instead of least squares, the parameters are found by maximising the likelihood of the observed labels (equivalently, minimising the log-loss / cross-entropy).
* We predict class 1 when $P > 0.5$, i.e. when $\mathbf{w}\cdot\mathbf{x} + b > 0$. That condition defines a straight line (a hyperplane in more dimensions): logistic regression is a linear classifier.
* Like Ridge regression, scikit-learn's `LogisticRegression` is regularized by default; the parameter `C` is the *inverse* of the regularization strength.

In [ ]:
t = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(t, 1/(1 + np.exp(-t)))
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.axvline(0, color="gray", ls="--", lw=1)
ax.set(xlabel=r"$t = \mathbf{w}\cdot\mathbf{x} + b$", ylabel=r"$\sigma(t)$", title="The sigmoid function");

### ✏️ Exercise 1 — Fit a logistic regression

1. Fit a `LogisticRegression` called `logreg` on the training set.
2. Compute the accuracy (fraction of correct predictions) on the test set, using `accuracy_score` or by hand.
3. Use `predict_proba` to get the probability of class 1 for the first five test objects.

In [ ]:
# YOUR CODE HERE
# logreg = 
# acc = 
# p_first5 = 


print(f"test accuracy: {acc:.3f}")
print("P(class 1) for the first five test objects:", np.round(p_first5, 3))
print("true classes                               :", y_test[:5])

In [ ]:
w, b = logreg.coef_[0], logreg.intercept_[0]
print(f"w = {np.round(w, 3)},  b = {b:.3f}")

fig, ax = plt.subplots()
plot_decision_boundary(logreg, X_test, y_test, ax,
                       title="Logistic regression: shading = P(class 1), line = 0.5")
ax.legend(loc="upper left");

## 2. Decision boundaries: linear vs. flexible

A linear classifier can only draw straight boundaries. Real classes are often not linearly separable. Here is a data set where a straight line cannot do well:

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=1)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.3, random_state=0, stratify=ym)

logreg_m = LogisticRegression().fit(Xm_train, ym_train)
fig, ax = plt.subplots()
plot_decision_boundary(logreg_m, Xm_train, ym_train, ax,
                       title=f"Logistic regression, test accuracy = "
                             f"{accuracy_score(ym_test, logreg_m.predict(Xm_test)):.2f}");

k-nearest neighbours (kNN) is one of the simplest flexible classifiers: to classify a new object, look at the k closest training objects and take a majority vote of their *labels*. There is no fitting at all — the training set *is* the model.

This is a different use of "nearest neighbours" than in the clustering notebook. There, `NearestNeighbors` was purely a distance tool — e.g. computing a local density $\Sigma_N$ from the distance to the Nth neighbour, or picking DBSCAN's `eps` from a k-distance plot. No labels were involved, because that data was unlabelled. Here the points *do* have labels (the two classes), so once we've found the k nearest neighbours we ask "what class are most of them?" — the same distance search, now used to predict a label instead of measuring density.

The number of neighbours k controls the flexibility, just like the polynomial degree did in the regression notebook:
* small k → very flexible, jagged boundary, follows the noise → overfitting
* large k → smooth boundary, eventually too simple → underfitting

### ✏️ Exercise 2 — Choosing k

For each `k` in `ks`, fit a `KNeighborsClassifier(n_neighbors=k)` on the moons training set and store the training and test accuracy in `acc_train` and `acc_test`. The next cells plot the result and the decision boundaries. Which k would you choose?

In [ ]:
ks = [1, 3, 5, 10, 20, 40, 80, 150]
acc_train, acc_test = [], []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xm_train, ym_train)
    acc_train.append(knn.score(Xm_train, ym_train))
    acc_test.append(knn.score(Xm_test, ym_test))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, acc_train, "o-", ms=7, label="training set")
ax.plot(ks, acc_test, "s-", ms=7, label="test set")
ax.set(xscale="log", xticks=ks, xticklabels=ks, xlabel="k  (number of neighbours)",
       ylabel="accuracy", title="kNN: flexibility decreases with k")
ax.legend();

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, k in zip(axes, [1, 20, 150]):
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xm_train, ym_train)
    plot_decision_boundary(knn, Xm_train, ym_train, ax,
                           title=f"k = {k}:  train {knn.score(Xm_train, ym_train):.2f},  "
                                 f"test {knn.score(Xm_test, ym_test):.2f}")
fig.tight_layout();

A support vector machine (SVM) is another flexible classifier, built on a different idea than kNN. For a linearly separable problem, it looks for the straight line that leaves the widest possible margin (gap) between the two classes; only the closest points on each side — the *support vectors* — determine where that line goes, everything else is ignored. The kernel trick lets it do the same thing on curved boundaries: an RBF kernel implicitly measures similarity to each training point in a way that behaves like mapping the data into a much higher-dimensional space, where a straight (max-margin) separator can correspond to a curved boundary back in the original 2-D plane.

Instead of a neighbour count, its `gamma` parameter sets how far a single training point's influence reaches: small `gamma` → influence reaches far → a smooth, almost-linear boundary; large `gamma` → influence is very local → the boundary can wrap tightly around individual points (overfitting). It plays a role similar to k, but in the opposite direction — small `gamma` is like large k.

### ✏️ Exercise — SVM: the kernel width `gamma`

Fit an `SVC(kernel="rbf", gamma=..., probability=True)` on the moons training set for `gamma` in 0.1, 1, 10 and compare the three boundaries to kNN's. Which `gamma` behaves most like which `k`?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, gamma in zip(axes, [0.1, 1, 10]):
    svm = SVC(kernel="rbf", gamma=gamma, probability=True).fit(Xm_train, ym_train)
    plot_decision_boundary(svm, Xm_train, ym_train, ax,
                           title=f"gamma = {gamma}:  train {svm.score(Xm_train, ym_train):.2f},  "
                                 f"test {svm.score(Xm_test, ym_test):.2f}")

fig.tight_layout();

## 3. Why accuracy is not enough

In astronomy the class we are interested in is often rare: high-redshift quasars among stars, real transients among image artifacts, strong lenses among ordinary galaxies. Let's build such a data set: 95% "background" objects and 5% "targets".

In [ ]:
Xi, yi = make_blobs(n_samples=[1900, 100], centers=[(0, 0), (2.2, 2.2)], cluster_std=[1.2, 0.9],
                    random_state=7)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.4, random_state=0, stratify=yi)
LABELS = ["background", "target"]

fig, ax = plt.subplots()
plot_data(Xi_train, yi_train, ax, LABELS, s=12, alpha=0.6)
ax.set_title(f"Imbalanced classes: {np.mean(yi_train == 1):.0%} targets")
ax.legend();

A "classifier" that always says background is right 95% of the time. Accuracy is useless here. We need to look at the four possible outcomes separately — the confusion matrix:

| | predicted background | predicted target |
|---|---|---|
| true background | true negative (TN) | false positive (FP) — *contamination* |
| true target | false negative (FN) — *missed* | true positive (TP) |

From it we define

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN} \qquad F_1 = 2\,\frac{\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}}$$

Astronomers already know these under other names:
* precision = purity: of the objects I selected, what fraction are real targets?
* recall = completeness: of all real targets, what fraction did I select?

$F_1$ is the harmonic mean of precision and recall. It is high only when *both* are reasonably high — a model that is excellent at one and terrible at the other still gets a low $F_1$ — which makes it a convenient single number for comparing classifiers, instead of having to eyeball two numbers (or four, as we will see with kNN, SVM and a random forest in the next notebook) at once.

In [ ]:
clf = LogisticRegression().fit(Xi_train, yi_train)
yi_pred = clf.predict(Xi_test)
always_bg = np.zeros_like(yi_test)

print(f"'always background' accuracy: {accuracy_score(yi_test, always_bg):.3f}")
print(f"logistic regression accuracy: {accuracy_score(yi_test, yi_pred):.3f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(yi_test, yi_pred, display_labels=LABELS, cmap="Blues",
                                        colorbar=False, ax=ax)
ax.grid(False)
ax.set_title("Logistic regression, test set");

### ✏️ Exercise 3 — Precision and recall by hand

Using only numpy and boolean comparisons of `yi_test` and `yi_pred`, compute `TP`, `FP`, `FN`, `TN`, and from them `precision` and `recall`. The cell checks your answer against scikit-learn.

What are the precision and recall of the "always background" classifier?

In [ ]:
# YOUR CODE HERE
# TP = 
# FP = 
# FN = 
# TN = 
# precision = 
# recall = 


print(f"TP = {TP}, FP = {FP}, FN = {FN}, TN = {TN}")
print(f"precision (purity)      = {precision:.3f}")
print(f"recall    (completeness) = {recall:.3f}")
assert np.isclose(precision, precision_score(yi_test, yi_pred))
assert np.isclose(recall, recall_score(yi_test, yi_pred))
print("matches scikit-learn ✓")

## 4. Choosing a threshold

`predict` uses a threshold of $P = 0.5$. Nothing forces us to. Lowering the threshold selects more objects: recall goes up, but so does contamination, and precision goes down. Which trade-off is right depends on the science case:

* Selecting candidates for expensive spectroscopic follow-up → we want high purity.
* Building a complete sample to measure a luminosity function → we want high completeness (and must correct for the contamination).

The precision–recall curve shows every possible trade-off at once.

In [ ]:
p_test = clf.predict_proba(Xi_test)[:, 1]
prec, rec, thr = precision_recall_curve(yi_test, p_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
ax1.plot(rec, prec, color="C0")
sel05 = p_test >= 0.5
ax1.plot(recall_score(yi_test, sel05), precision_score(yi_test, sel05), "o", ms=9, color="C1", label="threshold 0.5")
ax1.axhline(np.mean(yi_test), color="gray", ls="--", lw=1, label="random guessing")
ax1.set(xlabel="recall (completeness)", ylabel="precision (purity)", xlim=(0, 1.02), ylim=(0, 1.05),
        title="Precision–recall curve")
ax1.legend(loc="lower left")

ax2.plot(thr, prec[:-1], color="C0", label="precision")
ax2.plot(thr, rec[:-1], color="C1", label="recall")
ax2.set(xlabel="probability threshold", ylabel="score", xlim=(0, 1), ylim=(0, 1.05),
        title="Both depend on the threshold")
ax2.legend();

### ✏️ Exercise 4 — A complete sample

Your science case needs a sample that is at least 90% complete. Find the highest threshold `t90` for which the recall on the test set is still ≥ 0.9, and report the precision you get at that threshold. How many background objects contaminate your sample?

*Hint:* `thr`, `prec[:-1]` and `rec[:-1]` from the cell above are aligned arrays. Or loop over thresholds and use `recall_score(yi_test, p_test >= t)`.

In [ ]:
# YOUR CODE HERE
# ok = 
# t90 = 
# sel = 


print(f"threshold          : {t90:.3f}")
print(f"recall             : {recall_score(yi_test, sel):.3f}")
print(f"precision          : {precision_score(yi_test, sel):.3f}")
print(f"contaminants (FP)  : {np.sum(sel & (yi_test == 0))} of {sel.sum()} selected")

## 5. More than two classes

Many problems have more than two classes: star / galaxy / quasar, or morphological types. `LogisticRegression` generalises naturally: instead of one sigmoid it uses the softmax function to give one probability per class, summing to 1. Each pair of classes is still separated by a straight line, so the plane is divided into convex regions. kNN needs no change at all.

In [ ]:
LABELS3 = ["star", "galaxy", "quasar"]
X3, y3 = make_blobs(n_samples=[300, 300, 120], centers=[(-2, 0), (1.5, 1.5), (0.8, -1.2)],
                    cluster_std=[0.8, 1.0, 0.8], random_state=4)
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.4, random_state=0, stratify=y3)

### ✏️ Exercise 5 — A three-class classifier

1. Fit a `LogisticRegression` (`clf3`) on the three-class training set and predict `y3_pred` for the test set.
2. Look at the confusion matrix and the classification report below. Which two classes are most often confused? Which class has the lowest recall, and why might that be?

In [ ]:
# YOUR CODE HERE
# clf3 = 
# y3_pred = 


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[1.3, 1]))
plot_decision_boundary(clf3, X3_test, y3_test, ax1, labels=LABELS3, title="Decision regions (test set)")
ax1.legend(loc="upper left")
ConfusionMatrixDisplay.from_predictions(y3_test, y3_pred, display_labels=LABELS3, cmap="Blues",
                                        colorbar=False, ax=ax2)
ax2.grid(False)
ax2.set_title("Confusion matrix")
fig.tight_layout()

print(classification_report(y3_test, y3_pred, target_names=LABELS3, digits=3))

Reading the report

* Precision, recall and $F_1$ are computed per class, treating that class as "positive" and all others as "negative".
* The macro average gives every class equal weight; the weighted average weights by the number of objects in each class. With imbalanced classes the two can be very different — which one reflects how well you do on the *rare* class?

## Summary

* Logistic regression turns a linear model into class probabilities with the sigmoid (two classes) or softmax (more classes). Its decision boundaries are straight lines.
* Flexible classifiers such as kNN can draw curved boundaries, and overfit in exactly the same way as high-degree polynomials: judge them on the test set.
* Accuracy is misleading when classes are imbalanced. Use the confusion matrix, precision (purity) and recall (completeness).
* The probability threshold sets the trade-off between purity and completeness — choose it for your science case, not by default.